# 05 · Comparación en test — zero-shot frente a ajuste fino, entre modelos ($n = 10\,000$)

**Pregunta.** Sobre las mismas 10 000 imágenes de M3DI-*test*, ¿cómo se ordenan los modelos antes y después del ajuste fino? ¿Cuánto gana cada uno, en qué atributos y factores, y cambia el ranking de la etapa 1?

El notebook 04 analiza el ajuste **dentro** de cada modelo (curvas, validación, sondas). Aquí el foco es la comparación **entre** modelos y condiciones, siempre pareada sobre las mismas imágenes. Lee `runs/` en solo lectura y escribe en `reports/{figures,tables}/comparacion_test/`. Cada sección se salta con un aviso si sus runs todavía no existen.

| condición | qué es |
|---|---|
| **ZS p1** | zero-shot de la etapa 1 con `p1_attributes`, el mismo prompt con que se ajustó: aísla el efecto del ajuste |
| **ZS mejor** | zero-shot con el mejor prompt de cada modelo (índice macro, excluyendo corridas que copian los ejemplos del prompt). Se elige mirando test, como en la etapa 1: es una línea base **optimista**, así que la ganancia frente a ella es conservadora |
| **FT** | ajuste principal (objetivo canónico, train completo, NF4, semilla 0), checkpoint `adapter_best` elegido en val |
| FT composicional / objetivo original / bf16 / semilla 1 | controles de `stage2b_compositional` y `stage2b_ablations` (§8) |

Qwen3.5 Instruct aparece solo en zero-shot (control de la etapa 1, no se ajustó).

| § | Contenido |
|---|---|
| 1 | Configuración, inventario de condiciones e integridad (incluye el contraste con `test_metrics.json`) |
| 2 | Tabla general: todas las condiciones × todas las métricas |
| 3 | Panorama: ZS p1, ZS mejor y FT por modelo, con IC 95 % |
| 4 | Ganancia del ajuste por modelo y atributo (McNemar + Holm, brecha cerrada) |
| 5 | Comparación entre modelos: matrices pareadas y cambio de ranking |
| 6 | Desglose por forma, celda de posición y categoría de color |
| 7 | Aciertos por imagen: oráculo, errores compartidos e imágenes difíciles |
| 8 | Controles: split composicional, objetivo original, bf16, semilla |
| 9 | Síntesis y exportación |

**Notación.** Igual que el notebook 04: $k_{i,a}=\mathbb 1[\hat z_{i,a}=z_{i,a}]$ con el extractor de `vlmfid.eval.attributes` (omitir es fallar); `shape_pos` $=\tfrac12(k_\text{forma}+k_\text{pos})$, macro $=\tfrac13(k_\text{forma}+k_\text{pos}+k_{\text{color}\pm1})$, `macro_cat` con color exacto. Con el objetivo canónico el **nombre exacto es 0 por construcción**, así que se excluye de las comparaciones.

## 1. Configuración, inventario e integridad

In [ ]:
import json
import logging
import math
import re
import sys
import warnings
from itertools import combinations
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import Markdown, display

try:
    from vlmfid.paths import ROOT, RUNS
except ImportError:                      # kernel sin el paquete instalado en modo editable
    sys.path.insert(0, str(Path.cwd().resolve().parent / 'src'))
    from vlmfid.paths import ROOT, RUNS
from vlmfid.data import load_manifest, read_manifest
from vlmfid.data.targets import is_held_out
from vlmfid.eval import copy_flags, prompt_examples
from vlmfid.eval.attributes import COLOR_CATS, SHAPE_NAMES, XPOS_NAMES, YPOS_NAMES, extract, score, truth_from_record

# ── Parámetros ───────────────────────────────────────────────────────────────
SPLIT, VARIANT = 'test', 'base'
LIMIT = 10_000            # mismo conjunto que el zero-shot de la etapa 1 (data.limit=10000 o null)
SUBSET_SEED = 0
FT_PROMPT = 'p1_attributes'  # prompt del ajuste en los 4 modelos (y su línea base "ZS p1")
SELECT_METRIC = 'macro'   # criterio de la etapa 1 para elegir el mejor prompt zero-shot de cada modelo
MAX_COPY = 0.5            # corridas zero-shot con más copia de los ejemplos del prompt no compiten por "ZS mejor"
MIN_COVERAGE = 0.99       # ídem para corridas incompletas
RANK_METRIC = 'shape_pos' # métrica con que se comparan y ordenan los modelos (la de selección de la etapa 2)
ALPHA = 0.05
TAG = 'comparacion_test'
FIG_DIR = ROOT / 'reports' / 'figures' / TAG
TAB_DIR = ROOT / 'reports' / 'tables' / TAG
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

MODEL_LABELS = {                         # model.name -> etiqueta; el orden define el de figuras y tablas
    'vljepa_open_msrvtt': 'VL-JEPA',
    'qwen35_08b_base': 'Qwen3.5 Base',
    'internvl3_1b': 'InternVL3 1B',
    'llava_ov_05b': 'LLaVA-OV 0.5B',
    'qwen35_08b_instruct': 'Qwen3.5 Instruct',   # solo zero-shot: control de la etapa 1, no se ajustó
}
PROMPT_CATALOG = yaml.safe_load((ROOT / 'configs' / 'prompts.yaml').read_text(encoding='utf-8'))
PROMPT_SHORT = {p: ('p4m' if p.startswith('p4m') else p.split('_')[0]) for p in PROMPT_CATALOG}
JSON_PROMPTS = {p for p, c in PROMPT_CATALOG.items() if (c or {}).get('response_format') == 'json'}

FT_COND = {'principal': 'FT', 'composicional': 'FT composicional', 'original': 'FT objetivo original',
           'bf16': 'FT bf16', 'semilla': 'FT semilla 1'}   # clase del run de entrenamiento -> condición
COND_ORDER = ['ZS p1', 'ZS mejor', *FT_COND.values()]
COMPARE = ['ZS p1', 'ZS mejor', 'FT']   # las tres condiciones que se comparan entre todos los modelos
METRICS = ['shape', 'x', 'y', 'pos', 'color', 'color_adj', 'name', 'shape_pos', 'macro', 'macro_cat']
BINARY = ['shape', 'x', 'y', 'pos', 'color', 'color_adj', 'name']
TEST_METRICS = [m for m in METRICS if m != 'name']   # nombre exacto = 0 por construcción con el objetivo canónico
MLAB = {'shape': 'Forma', 'x': 'x', 'y': 'y', 'pos': 'Posición (x,y)', 'color': 'Color', 'color_adj': 'Color ±1',
        'name': 'Nombre exacto', 'shape_pos': 'Forma+pos', 'macro': 'Índice macro', 'macro_cat': 'Macro (color exacto)'}
MAIN_METRICS = ['shape', 'pos', 'color', 'color_adj', 'shape_pos', 'macro']

Z = 1.959963984540054
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 120)
warnings.filterwarnings('ignore', category=FutureWarning)
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
print(f'ROOT = {ROOT}\nRUNS = {RUNS}\nsalidas -> {FIG_DIR.relative_to(ROOT)}, {TAB_DIR.relative_to(ROOT)}')

In [ ]:
# ── Estilo (mismo sistema que los notebooks 02-04) y utilidades estadísticas ──
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']
INK, INK2, INK3, GRID, SURF, NEUTRAL = '#0b0b0b', '#52514e', '#8a8984', '#e6e5e1', '#fcfcfb', '#c9c8c2'
SEQ = mcolors.LinearSegmentedColormap.from_list(
    'seq_blue', ['#f4f8fe', '#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#184f95', '#0d366b'])
DIV = mcolors.LinearSegmentedColormap.from_list('div_red_blue', ['#c23a39', '#ee9190', '#f0efec', '#6da7ec', '#184f95'])
MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', '*']
MODEL_COLORS = {m: SERIES[i % len(SERIES)] for i, m in enumerate(MODEL_LABELS)}
MODEL_MARKERS = {m: MARKERS[i % len(MARKERS)] for i, m in enumerate(MODEL_LABELS)}
COND_STYLE = {'ZS p1': (NEUTRAL, 'o'), 'ZS mejor': (SERIES[1], 's'), 'FT': (SERIES[0], 'D')}

plt.rcParams.update({
    'figure.facecolor': SURF, 'axes.facecolor': SURF, 'savefig.facecolor': SURF,
    'axes.edgecolor': INK3, 'axes.labelcolor': INK2, 'axes.titlecolor': INK,
    'axes.titlesize': 10.5, 'axes.titleweight': 'semibold', 'axes.labelsize': 9.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'xtick.color': INK2, 'ytick.color': INK2, 'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
    'legend.frameon': False, 'legend.fontsize': 8.5, 'font.size': 9.5,
    'lines.linewidth': 2, 'lines.markersize': 7, 'figure.dpi': 110,
})
PCT = plt.FuncFormatter(lambda v, _: f'{v:.0%}')


def savefig(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(FIG_DIR / f'{name}.{ext}', dpi=200, bbox_inches='tight')


def save_table(df, name, index=True, float_format='%.3f'):
    df.to_csv(TAB_DIR / f'{name}.csv', index=index)
    try:
        df.to_latex(TAB_DIR / f'{name}.tex', index=index, float_format=float_format, escape=True, na_rep='--')
    except Exception as e:                       # to_latex requiere jinja2
        print(f'[aviso] {name}.tex no generado: {e}')


def heat(ax, M, xlabels, ylabels, fmt='{:.0%}', vmin=0.0, vmax=1.0, cmap=SEQ, fontsize=8, marks=None):
    """Mapa de calor anotado; `marks` (matriz booleana) agrega '*' a las celdas significativas."""
    M = np.asarray(M, dtype=float)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    ax.imshow(np.where(np.isfinite(M), M, np.nan), cmap=cmap, norm=norm, aspect='auto')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            if np.isfinite(v):
                r, g, b, _ = cmap(norm(v))
                lum = 0.2126 * r + 0.7152 * g + 0.0722 * b
                star = '*' if marks is not None and marks[i][j] else ''
                ax.text(j, i, fmt.format(v) + star, ha='center', va='center', fontsize=fontsize,
                        color='white' if lum < 0.5 else INK)
            else:
                ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, color=GRID, lw=0))
                ax.text(j, i, '–', ha='center', va='center', fontsize=fontsize, color=INK3)
    ax.set_xticks(range(M.shape[1]), xlabels)
    ax.set_yticks(range(M.shape[0]), ylabels)
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.tick_params(length=0)


def wilson(k, n, z=Z):
    k, n = np.asarray(k, float), np.asarray(n, float)
    with np.errstate(invalid='ignore', divide='ignore'):
        p = k / n
        den = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / den
        hw = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return c - hw, c + hw


def mcnemar_exact(b, c):
    # p-valor bilateral exacto de McNemar: bajo H0, b ~ Bin(b + c, 1/2).
    n = int(b + c)
    if n == 0:
        return 1.0
    k = int(min(b, c))
    logs = [math.lgamma(n + 1) - math.lgamma(i + 1) - math.lgamma(n - i + 1) - n * math.log(2) for i in range(k + 1)]
    m = max(logs)
    return min(1.0, 2 * math.exp(m + math.log(sum(math.exp(l - m) for l in logs))))


def holm(p):
    p = np.asarray(p, float)
    order, m = np.argsort(p), len(p)
    adj, run = np.empty(m), 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i])
        adj[i] = min(1.0, run)
    return adj


def read_jsonl(path):
    rows = []
    if not path.exists():
        return rows
    with path.open(encoding='utf-8') as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass                              # línea truncada por una caída
    return rows


def read_json(path):
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}


def read_yaml(path):
    return (yaml.safe_load(path.read_text(encoding='utf-8')) or {}) if path.exists() else {}


def skip(msg):
    print(f'[omitido] {msg}')

Runs de test que entran (mismo split, variante y 10 000 imágenes):

- **zero-shot** (`stage = zeroshot`): todos los prompts de la etapa 1; se separan NF4 y bf16.
- **ajustados** (`stage = finetuned` con `model.adapter_path`): el adaptador identifica su run de entrenamiento, y la clase (principal, composicional, controles, humo) se lee de la configuración de ese run, como en el notebook 04. Si `runs/train_*` no está (p. ej. se copió solo el run de test), la clase se deduce del nombre (`_hc`, `_torig`, `_bf16_`, `_s1`).

In [ ]:
def classify(cfg):
    """Clase de un run de entrenamiento según su config.yaml (igual que el notebook 04)."""
    d, t = cfg.get('data') or {}, cfg.get('train') or {}
    if d.get('val_limit') != 500 or t.get('max_steps'):
        return 'humo'
    fmt = (cfg.get('target') or {}).get('format', 'original')
    if d.get('holdout'):
        return 'composicional'
    if fmt == 'original':
        return 'original'
    if (cfg.get('model') or {}).get('quant') != 'nf4':
        return 'bf16'
    if int(cfg.get('seed') or 0) != 0:
        return 'semilla'
    return 'principal'


def train_kind(train_id):
    cfg = read_yaml(RUNS / train_id / 'config.yaml')
    if cfg:
        return classify(cfg)
    if re.search(r'_m\d+$', train_id):     # max_steps: prueba de humo
        return 'humo'
    for tag, kind in (('_hc', 'composicional'), ('_torig', 'original'), ('_bf16_', 'bf16'), ('_s1', 'semilla')):
        if tag in train_id:
            return kind
    return 'principal'


N_MAN = read_manifest(SPLIT, variant=VARIANT).height


def same_size(limit):
    # data.limit=10000 y data.limit=null son el mismo conjunto cuando el split tiene exactamente 10 000 imágenes.
    eff = N_MAN if limit in (None, 'null') else min(int(limit), N_MAN)
    return eff == min(LIMIT, N_MAN)


# ── Verdad de test ─────────────────────────────────────────────────────────────
records = load_manifest(SPLIT, limit=LIMIT, subset_seed=SUBSET_SEED, variant=VARIANT)
TRUTH = {r['id']: truth_from_record(r) for r in records}
truth = pd.DataFrame([{'id': r['id'], 'held_out': is_held_out(r), **TRUTH[r['id']]} for r in records]).set_index('id')
truth['cell'] = truth['y_true'] + '-' + truth['x_true']
EXPECTED = set(truth.index)

# ── Runs de test ───────────────────────────────────────────────────────────────
meta_rows, frames = [], []
for cfg_path in sorted(RUNS.glob('*/config.yaml')):
    cfg = read_yaml(cfg_path)
    d = cfg_path.parent
    data, mcfg, pcfg = cfg.get('data') or {}, cfg.get('model') or {}, cfg.get('prompt') or {}
    stage = cfg.get('stage')
    if stage not in ('zeroshot', 'finetuned') or data.get('split') != SPLIT or data.get('variant', 'base') != VARIANT:
        continue
    if not same_size(data.get('limit')) or mcfg.get('name') not in MODEL_LABELS:
        continue
    if stage == 'finetuned':
        if not mcfg.get('adapter_path'):
            continue
        train_id = Path(str(mcfg['adapter_path'])).parent.name
        kind = train_kind(train_id)
        if kind == 'humo':
            continue
    else:
        train_id, kind = None, ('zero-shot' if mcfg.get('quant') == 'nf4' else 'zero-shot bf16')
    st, tm = read_json(d / 'status.json'), read_json(d / 'test_metrics.json')
    preds = read_jsonl(d / 'predictions.jsonl')
    ids = [p['id'] for p in preds]
    meta_rows.append({'exp_id': cfg.get('exp_id', d.name), 'model': mcfg['name'], 'stage': stage, 'kind': kind,
                      'prompt': pcfg.get('id'), 'prompt_text': ' '.join(str(pcfg.get('text', '')).split()),
                      'train_id': train_id, 'estado': st.get('status', 'desconocido'), 'filas': len(ids),
                      'faltantes': len(EXPECTED - set(ids)), 'img_por_s': st.get('img_per_s'),
                      'tm_n': tm.get('n'), 'tm_macro': tm.get('macro'), 'tm_shape_pos': tm.get('shape_pos'),
                      'best_val_step': tm.get('best_val_step')})
    frames.append(pd.DataFrame({'exp_id': meta_rows[-1]['exp_id'], 'id': ids,
                                'prediction': [p.get('prediction') or '' for p in preds]}))

test_meta = pd.DataFrame(meta_rows)
HAVE_TEST = len(test_meta) > 0
if HAVE_TEST:
    test_meta['cobertura'] = 1 - test_meta['faltantes'] / len(EXPECTED)
    # dos runs de la misma configuración (p. ej. limit=10000 y null): se conserva el de mayor cobertura
    test_meta = (test_meta.sort_values('cobertura', ascending=False)
                 .drop_duplicates(['model', 'kind', 'prompt', 'train_id'], keep='first'))
    pred = pd.concat(frames, ignore_index=True)
    pred = pred[pred['exp_id'].isin(test_meta['exp_id']) & pred['id'].isin(EXPECTED)]
    pred = pred.drop_duplicates(['exp_id', 'id'], keep='last')
    pred = pred.merge(test_meta[['exp_id', 'model', 'kind', 'prompt']], on='exp_id')
    MODEL_ORDER = [m for m in MODEL_LABELS if m in set(test_meta['model'])]
    print(f'{SPLIT}: {len(truth):,} imágenes · {len(test_meta)} runs de test '
          f'({(test_meta["stage"] == "zeroshot").sum()} zero-shot, {(test_meta["stage"] == "finetuned").sum()} ajustados) '
          f'· {len(pred):,} predicciones')
else:
    MODEL_ORDER = []
    skip(f'no hay runs de test con n={LIMIT} en {RUNS}')
ML = MODEL_LABELS


def PS(p):
    return PROMPT_SHORT.get(p, p)

In [ ]:
# ── Puntaje por imagen (extractor de la etapa 1; p3_json se lee como JSON estricto) ──
_EXT = {}


def ext_of(text, json_mode):
    key = (text, json_mode)
    e = _EXT.get(key)
    if e is None:
        e = _EXT[key] = extract(text, json_mode)
    return e


FR = {}                   # (modelo, condición) -> predicciones puntuadas
COND = {}                 # (modelo, condición) -> exp_id
if HAVE_TEST:
    rows = []
    for i, t, p in zip(pred['id'], pred['prediction'], pred['prompt']):
        e = ext_of(t, p in JSON_PROMPTS)
        s = score(e, TRUTH[i])
        rows.append({**{k: float(v) for k, v in s.items()}, 'shape_pred': e['shape'] or 'omitido', 'n_words': e['n_words']})
    scored = pd.concat([pred.reset_index(drop=True), pd.DataFrame(rows)], axis=1).join(truth, on='id')

    # copia de los ejemplos del prompt (solo prompts con ejemplos concretos: p4, p4m)
    PROMPT_TEXT = dict(zip(test_meta['exp_id'], test_meta['prompt_text']))
    scored['copy_any'] = False
    for exp, idx in scored.groupby('exp_id').groups.items():
        exs = prompt_examples(PROMPT_TEXT[exp])
        if exs:
            g = scored.loc[idx]
            scored.loc[idx, 'copy_any'] = [bool(copy_flags(t, exs, s)['copy_any'])
                                           for t, s in zip(g['prediction'], g['shape_true'])]
    by_run = scored.groupby('exp_id')[METRICS + ['copy_any']].mean()
    by_run['n'] = scored.groupby('exp_id').size()
    META = test_meta.join(by_run, on='exp_id').set_index('exp_id')
    print(f'{len(scored):,} predicciones puntuadas · {len(_EXT):,} textos distintos extraídos')

    # ── Condiciones ──
    for m in MODEL_ORDER:
        t = META[META['model'] == m]
        z = t[t['kind'] == 'zero-shot']
        p1 = z[z['prompt'] == FT_PROMPT]
        if len(p1):
            COND[(m, 'ZS p1')] = p1.index[0]
        elig = z[(z['copy_any'] <= MAX_COPY) & (z['cobertura'] >= MIN_COVERAGE)]
        if len(elig):
            COND[(m, 'ZS mejor')] = elig[SELECT_METRIC].idxmax()
        for kind, c in FT_COND.items():
            f = t[t['kind'] == kind]
            if len(f):
                COND[(m, c)] = f.index[0]
    FR = {k: scored[scored['exp_id'] == e] for k, e in COND.items()}


def fr(m, c):
    return FR.get((m, c))


def summarize(df, metrics=METRICS):
    return {m: df[m].mean() for m in metrics} | {'n': len(df)}


def ci(df, k):
    """IC 95 %: Wilson para atributos binarios, normal para promedios por imagen."""
    n, v = len(df), df[k].mean()
    if k in BINARY:
        lo, hi = wilson(v * n, n)
        return float(lo), float(hi)
    se = df[k].std(ddof=1) / math.sqrt(n)
    return v - Z * se, v + Z * se


def paired(a, b, metric):
    """Comparación pareada B − A sobre las mismas imágenes. b = A acierta y B falla; c = A falla y B acierta."""
    m = a.set_index('id')[[metric]].join(b.set_index('id')[[metric]], how='inner', lsuffix='_a', rsuffix='_b')
    va, vb = m[f'{metric}_a'], m[f'{metric}_b']
    if metric in BINARY:
        b_, c_ = int(((va == 1) & (vb == 0)).sum()), int(((va == 0) & (vb == 1)).sum())
        p = mcnemar_exact(b_, c_)
    else:                                    # métricas promedio por imagen: z sobre la diferencia pareada
        diff = vb - va
        se = diff.std(ddof=1) / math.sqrt(len(diff)) if len(diff) > 1 else float('nan')
        p = math.erfc(abs(diff.mean() / se) / math.sqrt(2)) if se and se > 0 else 1.0
        b_ = c_ = np.nan
    return {'n': len(m), 'A': va.mean(), 'B': vb.mean(), 'Δ (B−A)': vb.mean() - va.mean(), 'b': b_, 'c': c_, 'p': p}


if COND:
    inv = pd.DataFrame([{'modelo': ML[m], 'condición': c,
                         'celda': f'{PS(META.loc[e, "prompt"])} · {META.loc[e, "cobertura"]:.0%}'
                                  + ('' if META.loc[e, 'estado'] == 'completed' else f' ({META.loc[e, "estado"]})')}
                        for (m, c), e in COND.items()])
    inv = inv.pivot(index='modelo', columns='condición', values='celda').reindex(
        index=[ML[m] for m in MODEL_ORDER], columns=[c for c in COND_ORDER if c in set(inv['condición'])])
    print('Condiciones disponibles (prompt · cobertura de las 10 000 imágenes):')
    display(inv.fillna('–'))
    inc = [e for e in COND.values() if META.loc[e, 'cobertura'] < 1]
    if inc:
        print(f'[aviso] {len(set(inc))} runs incompletos: sus métricas usan solo las imágenes ya inferidas y las '
              'comparaciones pareadas, la intersección.')
    zs_all = META[META['kind'] == 'zero-shot'].copy()
    zs_all['prompt'] = zs_all['prompt'].map(PS)
    zs_tab = zs_all.pivot_table(index='model', columns='prompt', values=SELECT_METRIC).rename(index=ML)
    print(f'\nZero-shot NF4, {MLAB[SELECT_METRIC]} por prompt (base para "ZS mejor"; '
          f'se excluyen corridas con copia > {MAX_COPY:.0%}):')
    display(zs_tab.style.format('{:.1%}', na_rep='–').highlight_max(axis=1, color='#cde2fb'))

In [ ]:
# ── Contraste con test_metrics.json (escrito por scripts/train.py al terminar cada ajuste) ──
if HAVE_TEST and META['tm_n'].notna().any():
    chk = META[META['tm_n'].notna()]
    chk = pd.DataFrame({'modelo': chk['model'].map(ML), 'clase': chk['kind'], 'paso val': chk['best_val_step'],
                        'n json': chk['tm_n'], 'n aquí': chk['n'],
                        'macro json': chk['tm_macro'], 'macro aquí': chk['macro'],
                        'forma+pos json': chk['tm_shape_pos'], 'forma+pos aquí': chk['shape_pos']})
    chk['|Δ| máx'] = np.maximum((chk['macro json'] - chk['macro aquí']).abs(),
                                (chk['forma+pos json'] - chk['forma+pos aquí']).abs())
    display(chk.style.format({c: '{:.4f}' for c in ['macro json', 'macro aquí', 'forma+pos json', 'forma+pos aquí']}
                             | {'|Δ| máx': '{:.1e}', 'n json': '{:,.0f}', 'n aquí': '{:,.0f}'}))
    bad = chk[(chk['|Δ| máx'] > 1e-6) | (chk['n json'] != chk['n aquí'])]
    if len(bad):
        print(f'[aviso] {len(bad)} runs no reproducen su test_metrics.json: revisar el extractor o el subconjunto.')
    else:
        print('✓ las métricas recalculadas coinciden con test_metrics.json en todos los runs ajustados.')
else:
    skip('sin test_metrics.json (runs entrenados con test.enabled=false o evaluados con stage2b_test.txt)')

## 2. Tabla general

Todas las condiciones disponibles, por modelo, sobre las mismas imágenes. El IC es de la métrica de comparación (`RANK_METRIC`, forma+pos por defecto). El mapa de calor resume las métricas principales; los bloques horizontales separan modelos.

In [ ]:
gen_rows = []
for m in MODEL_ORDER:
    for c in COND_ORDER:
        df = fr(m, c)
        if df is None:
            continue
        lo, hi = ci(df, RANK_METRIC)
        gen_rows.append({'modelo': ML[m], 'condición': c, 'prompt': PS(META.loc[COND[(m, c)], 'prompt']), **summarize(df),
                         f'IC {MLAB[RANK_METRIC]}': f'[{lo:.1%}, {hi:.1%}]'})
if gen_rows:
    general = pd.DataFrame(gen_rows).set_index(['modelo', 'condición'])
    display(general.style.format({k: '{:.1%}' for k in METRICS} | {'n': '{:,.0f}'}))
    save_table(general, 'tabla_general')

    fig, ax = plt.subplots(figsize=(1.05 * len(MAIN_METRICS) + 3, 0.34 * len(general) + 1.3))
    heat(ax, general[MAIN_METRICS].values, [MLAB[k] for k in MAIN_METRICS],
         [f'{a} · {b}' for a, b in general.index], fontsize=7.5)
    mods = general.index.get_level_values(0)
    for i in range(1, len(mods)):
        if mods[i] != mods[i - 1]:
            ax.axhline(i - 0.5, color=SURF, lw=3)
    ax.xaxis.tick_top()
    ax.set_title(f'Test (n = {len(truth):,}): todas las condiciones', pad=28)
    fig.tight_layout()
    savefig(fig, '01_tabla_general')
    plt.show()
else:
    skip('sin condiciones evaluadas en test')

## 3. Panorama: zero-shot frente a ajustado, por modelo

Para cada métrica, los tres puntos de cada modelo son ZS p1 (gris), ZS mejor (naranjo) y FT (azul), con IC 95 %. La flecha va de **ZS mejor** a **FT**: es la ganancia frente a la línea base más exigente.

In [ ]:
PLOT_METRICS = ['shape_pos', 'shape', 'pos', 'color', 'color_adj', 'macro']
mods = [m for m in MODEL_ORDER if any(fr(m, c) is not None for c in COMPARE)]
if mods:
    fig, axes = plt.subplots(1, len(PLOT_METRICS), figsize=(2.7 * len(PLOT_METRICS), 0.6 * len(mods) + 1.8), sharey=True)
    for ax, k in zip(axes, PLOT_METRICS):
        for j, c in enumerate(COMPARE):
            col, mk = COND_STYLE[c]
            pts = [(i + (j - 1) * 0.22, fr(m, c)) for i, m in enumerate(mods) if fr(m, c) is not None]
            if not pts:
                continue
            v = np.array([df[k].mean() for _, df in pts])
            cis = np.array([ci(df, k) for _, df in pts])
            ax.errorbar(v, [y for y, _ in pts], xerr=[v - cis[:, 0], cis[:, 1] - v], fmt=mk, color=col, ecolor=col,
                        capsize=2, ms=5, label=c, mec=INK3 if col == NEUTRAL else col)
        for i, m in enumerate(mods):
            a, b = fr(m, 'ZS mejor'), fr(m, 'FT')
            if a is not None and b is not None:
                ax.annotate('', xy=(b[k].mean(), i + 0.22), xytext=(a[k].mean(), i),
                            arrowprops=dict(arrowstyle='->', color=INK3, lw=0.8))
        ax.set_title(MLAB[k])
        ax.set_xlim(0, 1.02)
        ax.xaxis.set_major_formatter(PCT)
    axes[0].set_yticks(range(len(mods)), [ML[m] for m in mods])
    axes[0].invert_yaxis()
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc='lower center', ncol=len(COMPARE), bbox_to_anchor=(0.5, -0.02))
    fig.suptitle(f'Test (n = {len(truth):,}): zero-shot y ajustado por modelo', fontweight='semibold')
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    savefig(fig, '02_panorama')
    plt.show()
else:
    skip('sin condiciones ZS / FT en test')

## 4. Ganancia del ajuste

Comparación **pareada** FT − línea base, por modelo y métrica: McNemar exacto para atributos binarios ($b$ = aciertos perdidos, $c$ = ganados) y $z$ sobre la diferencia pareada para los promedios por imagen. Holm dentro de cada (línea base, modelo). La **brecha cerrada** $=\Delta / (1 - \text{base})$ es la fracción del error de la línea base que el ajuste elimina: permite comparar modelos que parten de niveles distintos. En el mapa, `*` = significativo tras Holm.

In [ ]:
gain_rows = []
for base in ('ZS p1', 'ZS mejor'):
    for m in MODEL_ORDER:
        a, b = fr(m, base), fr(m, 'FT')
        if a is None or b is None:
            continue
        rows = [{'línea base': base, 'modelo': ML[m], 'métrica': MLAB[k], 'k': k, **paired(a, b, k)} for k in TEST_METRICS]
        for r, p in zip(rows, holm([r['p'] for r in rows])):
            r['p_holm'], r['sig.'] = p, ('✓' if p < ALPHA else '')
        gain_rows += rows
if gain_rows:
    gains = pd.DataFrame(gain_rows)
    gains['brecha cerrada'] = gains['Δ (B−A)'] / (1 - gains['A'])
    gtab = gains.drop(columns='k').rename(columns={'A': 'base', 'B': 'FT'}).set_index(['línea base', 'modelo', 'métrica'])
    display(gtab.style.format({'base': '{:.1%}', 'FT': '{:.1%}', 'Δ (B−A)': '{:+.1%}', 'brecha cerrada': '{:+.0%}',
                               'p': '{:.1e}', 'p_holm': '{:.1e}', 'b': '{:,.0f}', 'c': '{:,.0f}', 'n': '{:,.0f}'}, na_rep='–'))
    save_table(gtab, 'ganancia_ajuste')

    bases = [b for b in ('ZS p1', 'ZS mejor') if b in set(gains['línea base'])]
    fig, axes = plt.subplots(1, len(bases), figsize=(7.2 * len(bases), 0.5 * gains['modelo'].nunique() + 2.2), squeeze=False)
    lim = max(0.05, gains['Δ (B−A)'].abs().max())
    for ax, base in zip(axes[0], bases):
        g = gains[gains['línea base'] == base]
        rows_g = list(dict.fromkeys(g['modelo']))
        D = g.pivot(index='modelo', columns='k', values='Δ (B−A)').reindex(index=rows_g, columns=TEST_METRICS)
        S = g.pivot(index='modelo', columns='k', values='p_holm').reindex(index=rows_g, columns=TEST_METRICS) < ALPHA
        heat(ax, D.values, [MLAB[k] for k in TEST_METRICS], rows_g, fmt='{:+.0%}', vmin=-lim, vmax=lim, cmap=DIV,
             fontsize=7.5, marks=S.values)
        ax.set_title(f'Δ = FT − {base}')
        ax.tick_params(axis='x', rotation=40)
        plt.setp(ax.get_xticklabels(), ha='right')
    fig.tight_layout()
    savefig(fig, '03_ganancia_ajuste')
    plt.show()
else:
    skip('sin pares (zero-shot, FT) del mismo modelo en test')

## 5. Comparación entre modelos

**Matrices pareadas.** Para cada condición, la celda (fila, columna) es $\bar M_\text{fila} - \bar M_\text{columna}$ de la métrica de comparación sobre las mismas imágenes; los modelos se ordenan por desempeño y `*` marca diferencias significativas (Holm sobre todos los pares de la condición). Si dos modelos ajustados no se separan, el ajuste no permite ordenarlos.

**Ranking.** El gráfico de pendientes muestra si el ordenamiento de la etapa 1 sobrevive al ajuste. Con el control de semilla (§8) se ve qué diferencia entre modelos es mayor que el ruido de entrenamiento.

In [ ]:
PAIR_METRICS = [RANK_METRIC, 'macro']
pair_conds = [c for c in COMPARE if sum(fr(m, c) is not None for m in MODEL_ORDER) >= 2]
pair_rows = []
RANKS = pd.DataFrame(index=[ML[m] for m in MODEL_ORDER], columns=COMPARE, dtype=float)
for c in COMPARE:
    vals = {ML[m]: fr(m, c)[RANK_METRIC].mean() for m in MODEL_ORDER if fr(m, c) is not None}
    if vals:
        RANKS[c] = pd.Series(vals).rank(ascending=False, method='min')

if pair_conds:
    fig, axes = plt.subplots(1, len(pair_conds), figsize=(4.8 * len(pair_conds), 4.4), squeeze=False)
    for ax, c in zip(axes[0], pair_conds):
        ms = sorted([m for m in MODEL_ORDER if fr(m, c) is not None], key=lambda m: -fr(m, c)[RANK_METRIC].mean())
        n = len(ms)
        D, P = np.full((n, n), np.nan), np.ones((n, n))
        rows_c = []
        for k in PAIR_METRICS:
            for i, j in combinations(range(n), 2):
                r = paired(fr(ms[j], c), fr(ms[i], c), k)          # Δ = B − A = fila − columna
                rows_c.append({'condición': c, 'métrica': MLAB[k], 'k': k, 'i': i, 'j': j, 'modelo 1': ML[ms[i]],
                               'modelo 2': ML[ms[j]], 'n': r['n'], '1': r['B'], '2': r['A'], 'Δ (1−2)': r['Δ (B−A)'],
                               'p': r['p']})
        for k in PAIR_METRICS:
            sel = [r for r in rows_c if r['k'] == k]
            for r, p in zip(sel, holm([r['p'] for r in sel])):
                r['p_holm'], r['sig.'] = p, ('✓' if p < ALPHA else '')
                if k == RANK_METRIC:
                    D[r['i'], r['j']], D[r['j'], r['i']] = r['Δ (1−2)'], -r['Δ (1−2)']
                    P[r['i'], r['j']] = P[r['j'], r['i']] = p
        pair_rows += rows_c
        lim = max(0.05, np.nanmax(np.abs(D)))
        heat(ax, D, [ML[m] for m in ms], [ML[m] for m in ms], fmt='{:+.1%}', vmin=-lim, vmax=lim, cmap=DIV,
             fontsize=7.5, marks=P < ALPHA)
        ax.set_title(f'{c}: Δ {MLAB[RANK_METRIC]} (fila − columna)')
        ax.tick_params(axis='x', rotation=30)
        plt.setp(ax.get_xticklabels(), ha='right')
    fig.tight_layout()
    savefig(fig, '04_matrices_pareadas')
    plt.show()
    ptab = pd.DataFrame(pair_rows).drop(columns=['k', 'i', 'j']).set_index(['condición', 'métrica', 'modelo 1', 'modelo 2'])
    display(ptab.style.format({'1': '{:.1%}', '2': '{:.1%}', 'Δ (1−2)': '{:+.1%}', 'p': '{:.1e}', 'p_holm': '{:.1e}',
                               'n': '{:,.0f}'}))
    save_table(ptab, 'pareadas_entre_modelos')
else:
    skip('menos de dos modelos por condición en test')

In [ ]:
if RANKS.notna().any().any():
    fig, ax = plt.subplots(figsize=(7, 4.6))
    for m in MODEL_ORDER:
        pts = [(x, fr(m, c)[RANK_METRIC].mean()) for x, c in enumerate(COMPARE) if fr(m, c) is not None]
        if not pts:
            continue
        xs, ys = zip(*pts)
        ax.plot(xs, ys, marker=MODEL_MARKERS[m], color=MODEL_COLORS[m], label=ML[m])
        ax.annotate(ML[m], (xs[-1], ys[-1]), xytext=(7, 0), textcoords='offset points', va='center', fontsize=8,
                    color=MODEL_COLORS[m])
    ax.set_xticks(range(len(COMPARE)), COMPARE)
    ax.set_xlim(-0.2, len(COMPARE) - 1 + 0.75)
    ax.yaxis.set_major_formatter(PCT)
    ax.set_ylabel(f'{MLAB[RANK_METRIC]} (test)')
    ax.set_title('¿Sobrevive el ranking de la etapa 1 al ajuste?')
    fig.tight_layout()
    savefig(fig, '05_pendientes_ranking')
    plt.show()

    rank_tab = pd.DataFrame(index=RANKS.index)
    for c in COMPARE:
        v = {ML[m]: fr(m, c)[RANK_METRIC].mean() for m in MODEL_ORDER if fr(m, c) is not None}
        rank_tab[c] = [f'{v[i]:.1%} (#{int(RANKS.loc[i, c])})' if i in v else '–' for i in RANKS.index]
    display(rank_tab)
    save_table(RANKS, 'ranking')

## 6. Desglose por factor

Exactitud de **forma** por clase verdadera, de **posición** $(x,y)$ por celda verdadera de la grilla 3×3 y de **color** (categoría exacta) por categoría verdadera, para ZS mejor y FT. El segundo panel muestra el cambio FT − ZS mejor: dónde se concentra la ganancia y si algún subgrupo empeora.

In [ ]:
BREAK_CONDS = ['ZS mejor', 'FT']
SHAPES = [SHAPE_NAMES[i] for i in range(7)]
CELLS = [f'{YPOS_NAMES[y]}-{XPOS_NAMES[x]}' for y in range(3) for x in range(3)]
FACTORS = [('Forma por clase', 'shape', 'shape_true', SHAPES), ('Posición por celda', 'pos', 'cell', CELLS),
           ('Color por categoría', 'color', 'cat_true', COLOR_CATS)]
keys = [(m, c) for m in MODEL_ORDER for c in BREAK_CONDS if fr(m, c) is not None]


def by_factor(df, k, col, cats):
    return df.groupby(col)[k].mean().reindex(cats).values


if keys:
    fig, axes = plt.subplots(1, 3, figsize=(20, 0.36 * len(keys) + 2.2), gridspec_kw={'width_ratios': [7, 9, 8]})
    long = []
    for j, (ax, (title, k, col, cats)) in enumerate(zip(axes, FACTORS)):
        M = np.array([by_factor(fr(m, c), k, col, cats) for m, c in keys])
        heat(ax, M, cats, [f'{ML[m]} · {c}' for m, c in keys], fontsize=6.5)
        if j:
            ax.set_yticks(range(len(keys)), [''] * len(keys))
        ax.set_title(title)
        ax.tick_params(axis='x', rotation=45)
        plt.setp(ax.get_xticklabels(), ha='right')
        for i in range(1, len(keys)):
            if keys[i][0] != keys[i - 1][0]:
                ax.axhline(i - 0.5, color=SURF, lw=3)
        long += [{'factor': title, 'modelo': ML[m], 'condición': c, 'clase': cat, 'exactitud': v}
                 for (m, c), row in zip(keys, M) for cat, v in zip(cats, row)]
    fig.tight_layout()
    savefig(fig, '06_desglose_factores')
    plt.show()
    save_table(pd.DataFrame(long).pivot_table(index=['factor', 'modelo', 'condición'], columns='clase',
                                              values='exactitud', sort=False), 'desglose_factores')

    dm = [m for m in MODEL_ORDER if fr(m, 'ZS mejor') is not None and fr(m, 'FT') is not None]
    if dm:
        fig, axes = plt.subplots(1, 3, figsize=(20, 0.42 * len(dm) + 2.2), gridspec_kw={'width_ratios': [7, 9, 8]})
        for j, (ax, (title, k, col, cats)) in enumerate(zip(axes, FACTORS)):
            M = np.array([by_factor(fr(m, 'FT'), k, col, cats) - by_factor(fr(m, 'ZS mejor'), k, col, cats) for m in dm])
            heat(ax, M, cats, [ML[m] for m in dm], fmt='{:+.0%}', vmin=-1, vmax=1, cmap=DIV, fontsize=7)
            if j:
                ax.set_yticks(range(len(dm)), [''] * len(dm))
            ax.set_title(f'{title}: FT − ZS mejor')
            ax.tick_params(axis='x', rotation=45)
            plt.setp(ax.get_xticklabels(), ha='right')
        fig.tight_layout()
        savefig(fig, '07_desglose_ganancia')
        plt.show()
else:
    skip('sin condiciones ZS mejor / FT en test')

## 7. Aciertos por imagen

Una imagen se cuenta como **acierto completo** si el modelo nombra bien la forma **y** la posición (`shape_pos` = 1). Se usan los modelos que tienen ZS mejor y FT, para que ambas condiciones tengan el mismo número de modelos.

- **Distribución** del número de modelos que aciertan cada imagen. Tras el ajuste, si los modelos convergen, la masa debería moverse a "todos".
- **Oráculo** (algún modelo acierta): techo de un ensamble ideal; su distancia al mejor modelo individual mide cuánto se complementan.
- **Solapamiento de errores** (Jaccard entre los conjuntos de imágenes falladas): valores altos = los modelos fallan en las mismas imágenes, es decir, la dificultad está en la imagen y no en el modelo.

In [ ]:
def hits(c, models):
    H = pd.concat({ML[m]: fr(m, c).set_index('id')['shape_pos'].eq(1) for m in models}, axis=1, join='inner')
    return H.astype(bool)


hm = [m for m in MODEL_ORDER if fr(m, 'ZS mejor') is not None and fr(m, 'FT') is not None]
if len(hm) >= 2:
    H = {c: hits(c, hm) for c in BREAK_CONDS}
    K = len(hm)
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.2), gridspec_kw={'width_ratios': [1.2, 1, 1]})
    ax = axes[0]
    for dx, c in ((-0.2, 'ZS mejor'), (0.2, 'FT')):
        cnt = H[c].sum(axis=1).value_counts(normalize=True).reindex(range(K + 1), fill_value=0)
        ax.bar(np.arange(K + 1) + dx, cnt.values, width=0.4, color=COND_STYLE[c][0], label=c)
    ax.set_xticks(range(K + 1))
    ax.set_xlabel(f'modelos (de {K}) que aciertan forma y posición')
    ax.yaxis.set_major_formatter(PCT)
    ax.set_title('Aciertos completos por imagen')
    ax.legend()
    for ax, c in zip(axes[1:], BREAK_CONDS):
        E = ~H[c]
        names = list(E.columns)
        J = np.array([[(E[a] & E[b]).sum() / max(1, (E[a] | E[b]).sum()) for b in names] for a in names])
        heat(ax, J, names, names, fontsize=8)
        ax.set_title(f'Solapamiento de errores (Jaccard) · {c}')
        ax.tick_params(axis='x', rotation=30)
        plt.setp(ax.get_xticklabels(), ha='right')
    fig.tight_layout()
    savefig(fig, '08_aciertos_por_imagen')
    plt.show()

    orc = pd.DataFrame({c: {'mejor modelo individual': H[c].mean().max(),
                            'mejor modelo': H[c].mean().idxmax(),
                            'algún modelo acierta (oráculo)': H[c].any(axis=1).mean(),
                            'todos aciertan': H[c].all(axis=1).mean(),
                            'ningún modelo acierta': (~H[c].any(axis=1)).mean(),
                            'n imágenes': len(H[c])} for c in BREAK_CONDS})
    display(orc)
    save_table(orc, 'oraculo')

    # imágenes que ningún modelo ajustado acierta: ¿se concentran en alguna forma o celda?
    hard = truth.loc[H['FT'].index[~H['FT'].any(axis=1)]]
    base = truth.loc[H['FT'].index]
    if len(hard):
        parts = {}
        for name, col in (('forma', 'shape_true'), ('celda', 'cell')):
            t = pd.DataFrame({'% de las difíciles': hard[col].value_counts(normalize=True),
                              '% en test': base[col].value_counts(normalize=True)}).fillna(0)
            t['sobrerrepresentación'] = t['% de las difíciles'] / t['% en test']
            parts[name] = t.sort_values('sobrerrepresentación', ascending=False)
        hard_tab = pd.concat(parts)
        print(f'{len(hard):,} imágenes ({len(hard) / len(base):.1%}) que ningún modelo ajustado acierta:')
        display(hard_tab.style.format({'% de las difíciles': '{:.1%}', '% en test': '{:.1%}',
                                       'sobrerrepresentación': '{:.2f}×'}))
        save_table(hard_tab, 'imagenes_dificiles_ft')
else:
    skip('menos de dos modelos con ZS mejor y FT en test')

## 8. Controles

Resumen de los controles frente al FT principal del mismo modelo (el detalle está en el notebook 04, §6 y §8). Pareado sobre las mismas imágenes, Holm dentro de cada (control, modelo).

- **Composicional**: además, forma+pos en las combinaciones forma × posición **vistas** y **retiradas** del train; la brecha (vistas − retiradas) mide memorización de pares. El zero-shot sirve de control: no debería tener brecha.
- **Semilla 1 − semilla 0**: ruido de entrenamiento. Una diferencia entre modelos (§5) menor que esta no es interpretable.

In [ ]:
CTRL_CONDS = COND_ORDER[3:]
ctrl_rows = []
for m in MODEL_ORDER:
    a = fr(m, 'FT')
    if a is None:
        continue
    for c in CTRL_CONDS:
        b = fr(m, c)
        if b is None:
            continue
        rows = [{'control': c, 'modelo': ML[m], 'métrica': MLAB[k], **paired(a, b, k)}
                for k in ('shape_pos', 'shape', 'pos', 'color', 'macro')]
        for r, p in zip(rows, holm([r['p'] for r in rows])):
            r['p_holm'], r['sig.'] = p, ('✓' if p < ALPHA else '')
        ctrl_rows += rows
if ctrl_rows:
    ctrl = pd.DataFrame(ctrl_rows).rename(columns={'A': 'FT', 'B': 'control', 'Δ (B−A)': 'Δ (control − FT)'})
    ctrl = ctrl.set_index(['control', 'modelo', 'métrica'])
    display(ctrl.style.format({'FT': '{:.1%}', 'control': '{:.1%}', 'Δ (control − FT)': '{:+.1%}', 'p': '{:.1e}',
                               'p_holm': '{:.1e}', 'b': '{:,.0f}', 'c': '{:,.0f}', 'n': '{:,.0f}'}, na_rep='–'))
    save_table(ctrl, 'controles')
else:
    skip('sin controles evaluados en test (stage2b_compositional / stage2b_ablations)')

comp_rows = []
for m in MODEL_ORDER:
    if fr(m, 'FT composicional') is None:
        continue
    for c in ('ZS p1', 'FT', 'FT composicional'):
        df = fr(m, c)
        if df is None:
            continue
        seen, held = df.loc[~df['held_out'], 'shape_pos'].mean(), df.loc[df['held_out'], 'shape_pos'].mean()
        comp_rows.append({'modelo': ML[m], 'condición': c, 'vistas': seen, 'retiradas': held, 'brecha': seen - held})
if comp_rows:
    comp = pd.DataFrame(comp_rows).set_index(['modelo', 'condición'])
    print(f'{MLAB["shape_pos"]} en combinaciones vistas y retiradas del split composicional:')
    display(comp.style.format({'vistas': '{:.1%}', 'retiradas': '{:.1%}', 'brecha': '{:+.1%}'}))
    save_table(comp, 'composicional_resumen')

## 9. Síntesis y exportación

In [ ]:
synth = []
for m in MODEL_ORDER:
    row = {'modelo': ML[m]}
    if (m, 'ZS mejor') in COND:
        row['prompt ZS mejor'] = PS(META.loc[COND[(m, 'ZS mejor')], 'prompt'])
    for c in COMPARE:
        df = fr(m, c)
        row[f'{MLAB[RANK_METRIC]} {c}'] = df[RANK_METRIC].mean() if df is not None else np.nan
    for c in ('ZS mejor', 'FT'):
        df = fr(m, c)
        row[f'macro {c}'] = df['macro'].mean() if df is not None else np.nan
    zs, ft = row[f'{MLAB[RANK_METRIC]} ZS mejor'], row[f'{MLAB[RANK_METRIC]} FT']
    row['Δ FT − ZS mejor'] = ft - zs
    row['brecha cerrada'] = (ft - zs) / (1 - zs) if np.isfinite(zs) and zs < 1 else np.nan
    if ML[m] in RANKS.index:
        row['puesto ZS mejor'] = RANKS.loc[ML[m], 'ZS mejor']
        row['puesto FT'] = RANKS.loc[ML[m], 'FT']
    synth.append(row)
if synth:
    synth = pd.DataFrame(synth).set_index('modelo').dropna(axis=1, how='all')
    pct = [c for c in synth.columns if c not in ('prompt ZS mejor', 'puesto ZS mejor', 'puesto FT')]
    display(synth.style.format({c: '{:.1%}' for c in pct} | {'Δ FT − ZS mejor': '{:+.1%}', 'brecha cerrada': '{:+.0%}',
                                                            'puesto ZS mejor': '{:.0f}', 'puesto FT': '{:.0f}'}, na_rep='–'))
    save_table(synth, 'sintesis')

    # ── Lectura automática (cifras de esta ejecución) ──
    lab = MLAB[RANK_METRIC]
    zcol, fcol = f'{lab} ZS mejor', f'{lab} FT'
    lines = []
    if zcol in synth and synth[zcol].notna().any():
        b = synth[zcol].idxmax()
        lines.append(f'- **Mejor zero-shot:** {b} ({synth.loc[b, zcol]:.1%} {lab}, prompt {synth.loc[b, "prompt ZS mejor"]}).')
    if fcol in synth and synth[fcol].notna().any():
        b = synth[fcol].idxmax()
        lines.append(f'- **Mejor ajustado:** {b} ({synth.loc[b, fcol]:.1%}).')
        g = synth['Δ FT − ZS mejor'].dropna()
        if len(g):
            lines.append(f'- **Mayor ganancia frente a ZS mejor:** {g.idxmax()} ({g.max():+.1%}; '
                         f'cierra {synth.loc[g.idxmax(), "brecha cerrada"]:.0%} del error); menor: {g.idxmin()} ({g.min():+.1%}).')
        both = synth[['puesto ZS mejor', 'puesto FT']].dropna() if 'puesto FT' in synth else pd.DataFrame()
        if len(both):
            same = (both['puesto ZS mejor'].rank() == both['puesto FT'].rank()).all()
            lines.append('- **Ranking:** ' + ('se mantiene tras el ajuste.' if same else
                         'cambia tras el ajuste: ' + ', '.join(f'{i} #{int(r["puesto ZS mejor"])}→#{int(r["puesto FT"])}'
                                                              for i, r in both.iterrows()) + '.'))
    if ctrl_rows:
        s = pd.DataFrame(ctrl_rows)
        s = s[(s['control'] == 'FT semilla 1') & (s['métrica'] == lab)]
        if len(s):
            lines.append(f'- **Ruido de semilla** en {lab}: |Δ| ≤ {s["Δ (B−A)"].abs().max():.1%} '
                         f'({", ".join(s["modelo"])}). Diferencias entre modelos menores que esto no son interpretables.')
    if lines:
        display(Markdown('\n'.join(lines)))
print(f'Tablas en {TAB_DIR.relative_to(ROOT)} · figuras en {FIG_DIR.relative_to(ROOT)}')